# semantic_search/02 — Principal components within note types

Computes patient PCs separately for clinician, imaging, and pathology embeddings using **row L2 normalization → feature standardization → PCA**.

**Runs after** `01_aggregate` and before `03_pc_correlations`. Compatible artifacts are reused unless `OVERWRITE = True`.

In [ ]:
from __future__ import annotations

import os, subprocess, sys, time
from pathlib import Path
import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.py').is_file() and (candidate / 'semantic_search').is_dir():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from semantic_search import common

def run_module(module, args):
    cmd = [sys.executable, '-m', module, *args]
    print('$ ' + ' '.join(cmd) + '\n', flush=True)
    started = time.time()
    result = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f'\nexit={result.returncode}  elapsed={(time.time() - started) / 60:,.1f} min')
    return result.returncode

SPACES = common.PC_SPACES
WINDOWS = common.DEFAULT_WINDOWS
N_COMPONENTS = 50
SEED = 0
OVERWRITE = False
RUN_PCA = True

print(f'repo root:  {REPO_ROOT}')
print(f'spaces:     {SPACES}')
print(f'windows:    {WINDOWS}')
print(f'components: {N_COMPONENTS}')

## Pre-flight and run

In [ ]:
missing = []
for window in WINDOWS:
    for space in SPACES:
        path = common.feature_path(space, window)
        exists = os.path.exists(path)
        print(f"[{'ok ' if exists else 'MISSING'}] {space}/{window}: {path}")
        if not exists:
            missing.append(path)

if RUN_PCA and not missing:
    args = ['--spaces', *SPACES, '--windows', *WINDOWS, '--n-components', str(N_COMPONENTS), '--seed', str(SEED)]
    if OVERWRITE:
        args.append('--overwrite')
    return_code = run_module('semantic_search.compute_pcs', args)
    if return_code:
        raise RuntimeError(f'PCA stage exited with {return_code}')
elif missing:
    print('Run 01_aggregate first.')
else:
    print('RUN_PCA=False; skipped.')

## Explained variance

In [ ]:
variance_path = common.result_path('pc_explained_variance')
if os.path.exists(variance_path):
    variance = pl.read_csv(variance_path).filter(pl.col('space').is_in(SPACES) & pl.col('window').is_in(WINDOWS))
    display(variance.head(10))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for key, frame in variance.group_by(['space', 'window'], maintain_order=True):
        label = '/'.join(key)
        axes[0].plot(frame['component'], 100 * frame['explained_variance_ratio'], marker='o', ms=3, label=label)
        axes[1].plot(frame['component'], 100 * frame['cumulative_explained_variance_ratio'], marker='o', ms=3, label=label)
    axes[0].set(xlabel='principal component', ylabel='% variance', title='Scree plot')
    axes[1].set(xlabel='principal component', ylabel='cumulative % variance', title='Cumulative variance')
    axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()
else:
    print(f'No result at {variance_path}')

## Next

Run `03_pc_correlations.ipynb` for clinical association tests, or `04_predict.ipynb` for supervised label prediction.